# 🥉 ETL: Raw → Bronze
## Crime Data from 2020 to Present - Los Angeles

### Arquitetura Medalhão - Camada Bronze

Este notebook realiza a extração e carga dos dados brutos (Raw) para a camada Bronze.

**Camada Bronze**: Dados brutos, tal como extraídos da fonte original, com adição de metadados de ingestão.

### Etapas:
1. Extração dos dados do arquivo CSV
2. Validação básica da estrutura
3. Adição de metadados de ingestão
4. Persistência na camada Bronze

In [ ]:
# Importações
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path
import os
import warnings
warnings.filterwarnings('ignore')

# Configurações
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print('✅ Bibliotecas carregadas com sucesso!')
print(f'📅 Data de execução: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

## 1. Configuração de Caminhos e Parâmetros

In [ ]:
# Configuração de diretórios (robusto a diferentes CWDs)
def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / 'SBD2', cwd.parent]
    for root in candidates:
        if (root / 'Crime_Data_from_2020_to_Present.csv').exists():
            return root
    return cwd

PROJECT_ROOT = find_project_root()
RAW_DATA_PATH = PROJECT_ROOT / 'Crime_Data_from_2020_to_Present.csv'
BRONZE_DATA_DIR = PROJECT_ROOT / 'data' / 'bronze'
BRONZE_OUTPUT_PATH = BRONZE_DATA_DIR / 'crime_data_bronze.parquet'

# Criar diretório Bronze se não existir
BRONZE_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Parâmetros de ingestão
BATCH_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
SOURCE_SYSTEM = 'LAPD_Crime_Data'

print(f'📁 Projeto: {PROJECT_ROOT}')
print(f'📁 Raw: {RAW_DATA_PATH}')
print(f'📁 Bronze: {BRONZE_DATA_DIR}')
print(f'🔖 Batch ID: {BATCH_ID}')

## 2. Extração dos Dados Brutos (Raw)

In [ ]:
# Verificar se arquivo existe
if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {RAW_DATA_PATH}")

print(f"📂 Carregando dados de: {RAW_DATA_PATH}")
print(f"📊 Tamanho do arquivo: {RAW_DATA_PATH.stat().st_size / 1024**2:.2f} MB")

In [ ]:
# Carregar dados brutos
# Dica: para testes rápidos, defina NROWS no ambiente (ex.: NROWS=50000)
nrows = int(os.getenv('NROWS', '0'))
df_raw = pd.read_csv(RAW_DATA_PATH, nrows=(nrows if nrows > 0 else None))

print(f"\n✅ Dados carregados com sucesso!")
print(f"📊 Shape: {df_raw.shape[0]:,} linhas x {df_raw.shape[1]} colunas")
print(f"💾 Memória utilizada: {df_raw.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
# Visualizar estrutura dos dados brutos
print("📋 Primeiras linhas do dataset:")
df_raw.head()

In [ ]:
# Informações sobre o dataset
print("📊 Informações do Dataset:")
df_raw.info()

## 3. Validação Básica da Estrutura

In [ ]:
# Colunas esperadas do dataset
EXPECTED_COLUMNS = [
    'DR_NO', 'Date Rptd', 'DATE OCC', 'TIME OCC', 'AREA', 'AREA NAME',
    'Rpt Dist No', 'Part 1-2', 'Crm Cd', 'Crm Cd Desc', 'Mocodes',
    'Vict Age', 'Vict Sex', 'Vict Descent', 'Premis Cd', 'Premis Desc',
    'Weapon Used Cd', 'Weapon Desc', 'Status', 'Status Desc',
    'Crm Cd 1', 'Crm Cd 2', 'Crm Cd 3', 'Crm Cd 4', 'LOCATION',
    'Cross Street', 'LAT', 'LON'
]

# Validar colunas presentes
missing_cols = set(EXPECTED_COLUMNS) - set(df_raw.columns)
extra_cols = set(df_raw.columns) - set(EXPECTED_COLUMNS)

print("🔍 Validação de Colunas:")
print(f"   ✅ Colunas esperadas: {len(EXPECTED_COLUMNS)}")
print(f"   📋 Colunas presentes: {len(df_raw.columns)}")

if missing_cols:
    print(f"   ⚠️ Colunas faltantes: {missing_cols}")
else:
    print("   ✅ Todas as colunas esperadas estão presentes")

if extra_cols:
    print(f"   ℹ️ Colunas extras: {extra_cols}")

In [ ]:
# Validar chave primária (DR_NO - Division of Records Number)
total_records = len(df_raw)
unique_ids = df_raw['DR_NO'].nunique()
duplicates = total_records - unique_ids

print("🔑 Validação de Chave Primária (DR_NO):")
print(f"   📊 Total de registros: {total_records:,}")
print(f"   🔢 IDs únicos: {unique_ids:,}")
print(f"   🔄 Duplicatas: {duplicates:,}")

if duplicates > 0:
    print(f"   ⚠️ Atenção: {duplicates:,} registros duplicados encontrados")
else:
    print("   ✅ Sem duplicatas na chave primária")

In [ ]:
# Estatísticas de valores nulos
null_counts = df_raw.isnull().sum()
null_percentage = (null_counts / len(df_raw) * 100).round(2)

null_stats = pd.DataFrame({
    'Valores Nulos': null_counts,
    'Porcentagem (%)': null_percentage
}).sort_values('Porcentagem (%)', ascending=False)

print("📊 Estatísticas de Valores Nulos:")
null_stats[null_stats['Valores Nulos'] > 0]

## 4. Adição de Metadados de Ingestão (Bronze Layer)

In [ ]:
# Criar DataFrame Bronze com metadados
df_bronze = df_raw.copy()

# Adicionar metadados de ingestão
df_bronze['_ingestion_timestamp'] = datetime.now()
df_bronze['_batch_id'] = BATCH_ID
df_bronze['_source_system'] = SOURCE_SYSTEM
df_bronze['_source_file'] = RAW_DATA_PATH.name

# Hash estável e rápido (evita .apply linha-a-linha e hash não determinístico)
df_bronze['_row_hash'] = pd.util.hash_pandas_object(df_raw, index=False).astype('uint64')

print("✅ Metadados de ingestão adicionados:")
print(f"   🕐 _ingestion_timestamp: Timestamp da ingestão")
print(f"   🔖 _batch_id: {BATCH_ID}")
print(f"   📂 _source_system: {SOURCE_SYSTEM}")
print(f"   📄 _source_file: {RAW_DATA_PATH.name}")
print(f"   🔐 _row_hash: Hash para detecção de mudanças")

In [ ]:
# Verificar estrutura final do DataFrame Bronze
print("📊 Estrutura da Camada Bronze:")
print(f"   Shape: {df_bronze.shape[0]:,} linhas x {df_bronze.shape[1]} colunas")
print(f"\n📋 Colunas adicionadas (metadados):")
metadata_cols = [col for col in df_bronze.columns if col.startswith('_')]
for col in metadata_cols:
    print(f"   - {col}")

In [ ]:
# Amostra dos dados Bronze
print("📋 Amostra dos dados Bronze (com metadados):")
df_bronze[['DR_NO', 'DATE OCC', 'AREA NAME', 'Crm Cd Desc', '_ingestion_timestamp', '_batch_id']].head()

## 5. Persistência na Camada Bronze

In [ ]:
# Salvar como Parquet (formato otimizado para Data Lake)
used_compression = 'snappy'
try:
    df_bronze.to_parquet(BRONZE_OUTPUT_PATH, index=False, compression=used_compression)
except Exception as e:
    print(f"⚠️ Falha ao salvar com '{used_compression}'. Tentando 'gzip'. Erro: {e}")
    used_compression = 'gzip'
    df_bronze.to_parquet(BRONZE_OUTPUT_PATH, index=False, compression=used_compression)

print(f"✅ Dados salvos na camada Bronze!")
print(f"   📁 Caminho: {BRONZE_OUTPUT_PATH}")
print(f"   📦 Compressão: {used_compression}")
print(f"   📊 Registros: {len(df_bronze):,}")
print(f"   💾 Tamanho: {BRONZE_OUTPUT_PATH.stat().st_size / 1024**2:.2f} MB")

In [ ]:
# Salvar também em CSV para backup/compatibilidade
csv_path = BRONZE_DATA_DIR / 'crime_data_bronze.csv'
df_bronze.to_csv(csv_path, index=False)

print(f"✅ Backup CSV salvo:")
print(f"   📁 Caminho: {csv_path}")
print(f"   💾 Tamanho: {csv_path.stat().st_size / 1024**2:.2f} MB")

In [ ]:
# Criar arquivo de metadados da ingestão
import json

ingestion_metadata = {
    'batch_id': BATCH_ID,
    'source_system': SOURCE_SYSTEM,
    'source_file': RAW_DATA_PATH.name,
    'ingestion_timestamp': datetime.now().isoformat(),
    'total_records': len(df_bronze),
    'total_columns': len(df_bronze.columns),
    'columns': list(df_bronze.columns),
    'null_statistics': null_stats.to_dict(),
    'duplicate_count': duplicates,
    'output_format': 'parquet',
    'compression': used_compression,
}

metadata_path = BRONZE_DATA_DIR / f'ingestion_metadata_{BATCH_ID}.json'
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(ingestion_metadata, f, indent=2, default=str)

print(f"✅ Metadados de ingestão salvos:")
print(f"   📁 Caminho: {metadata_path}")

## 6. Verificação e Qualidade dos Dados

In [ ]:
# Verificar leitura do arquivo Parquet
try:
    df_verify = pd.read_parquet(BRONZE_OUTPUT_PATH)
    print("🔍 Verificação de Integridade:")
    print(f"   📊 Registros originais: {len(df_bronze):,}")
    print(f"   📊 Registros verificados: {len(df_verify):,}")
    print(f"   ✅ Integridade: {'OK' if len(df_bronze) == len(df_verify) else 'FALHA'}")
except Exception as e:
    raise RuntimeError(
        "Falha ao ler o Parquet gerado. "
        "Instale 'pyarrow' (recomendado) ou 'fastparquet'. "
        f"Erro: {e}"
    )

In [ ]:
# Resumo da camada Bronze
print("=" * 60)
print("📊 RESUMO DA INGESTÃO - CAMADA BRONZE")
print("=" * 60)
print(f"\n🔖 Batch ID: {BATCH_ID}")
print(f"📅 Data/Hora: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"\n📊 Estatísticas:")
print(f"   • Total de registros: {len(df_bronze):,}")
print(f"   • Total de colunas: {len(df_bronze.columns)}")
print(f"   • Colunas originais: {len(df_raw.columns)}")
print(f"   • Colunas de metadados: {len(metadata_cols)}")
print(f"\n📁 Arquivos gerados:")
print(f"   • {BRONZE_OUTPUT_PATH}")
print(f"   • {csv_path}")
print(f"   • {metadata_path}")
print("\n✅ ETL Raw → Bronze concluído com sucesso!")

## 📋 Resumo - ETL Raw → Bronze

### O que foi realizado:
- ✅ Extração dos dados brutos do arquivo CSV
- ✅ Validação da estrutura (colunas, tipos, chave primária)
- ✅ Detecção de valores nulos e duplicatas
- ✅ Adição de metadados de ingestão:
  - Timestamp de ingestão
  - ID do batch
  - Sistema fonte
  - Arquivo fonte
  - Hash de linha para CDC
- ✅ Persistência em formato Parquet (otimizado)
- ✅ Backup em formato CSV
- ✅ Geração de metadados da ingestão

### Próximo passo:
Execute o notebook `02_bronze_to_silver.ipynb` para transformar os dados da camada Bronze para Silver.